# Problem Statement 
Build me an agentic QnA system that can answer me questions from excel file or general queries or custom mathematical questions.

### Architecture

1. we take the user query and the document 
2. if the user has a general query or a mathematical question - no reference given - we direclty ask the LLM to answer 
3. if the user has a document - we take that document and the user query and pass it to LLM 
   - now LLM1 - will be given the user query and the dataschema (column name and the data types) or the first 10 rows for context - this LLM will generate a pandas code for us
   - now that we have the pandas code - we will validate this with another LLM, LLM2. 3 retries allowed
   - if validation passes - we execute the code in a sandbox environment 
   - LLM3 will give the result to the user
   - if the retries fail, we ask LLM3 to give the user a negative reply that we are not able to generate a answer. 
4. intent classifier checks if we need to send the query to directly LLM3 or go through the pipeline to extract code, validate and then execute


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("Loaded GROQ_API_KEY", groq_api_key[:8])
else:
    print("GROQ API KEY not found")

Loaded GROQ_API_KEY gsk_wrLm


### Step 2 Intent Classifier

In [5]:
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print(client)

def ask_llm(user_query: str, system_prompt: str = "You are a helpful assistant.") -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages= [
            { "role": "system","content": system_prompt },  
            { "role": "user", "content": user_query }                
        ]
    )
    # print(f"Question: {user_query}: \nAnswer: {response.choices[0].message.content}")
    return response.choices[0].message.content
    
# test it
# print(ask_llm("What is 12 * 8?", system_prompt="You only answer in French."))
# print(ask_llm("What's the capital of France?"))

In [6]:
def classify_intent(user_query: str, has_document: bool) -> str:
    system_prompt = """
    You are an intent classifier. Given a user query, respond with EXACTLY one word:
    - "direct" if the query is general knowledge, math, or doesn't need a document
    - "document" if the query requires looking at uploaded document data
    
    Respond with only the single word, nothing else.
    """
    
    user_content = f"has_document: {has_document}\nquery: {user_query}"
    
    raw_response = ask_llm(user_content, system_prompt)
    
    cleaned = raw_response.strip().lower()
    
    return cleaned

# test cases
# print(classify_intent("What is 12 * 8?", has_document=False))
# print(classify_intent("What's the capital of France?", has_document=False))
# print(classify_intent("What's the average revenue in column C?", has_document=True))

### Step 3: Router

In [7]:
import pandas as pd

def load_document(file_path: str):
    df = pd.read_excel(file_path)
    # print(df)
    return df

def get_schema_context(df) -> str:
    sample_rows = df.head().to_string()
    
    schema_context = f"""
    Sample data from the document:
    {sample_rows}
    """
    return schema_context

### Pandas code generator LLM

In [8]:
def generate_pandas_code(user_query: str, schema_context: str, previous_error: str = None) -> str:
    system_prompt = """
    You are a pandas code generator. Given a user query and sample data from a DataFrame called `df`,
    write ONE line (or a few lines) of pandas code that answers the query.
    
    Rules:
    - Assume the DataFrame is already loaded as `df` — do not include import statements or df creation
    - Store the final answer in a variable called `result`
    - Output ONLY the code, no explanation, no markdown code fences, no commentary
    """
    
    if previous_error:  
        user_content = f"Schema/sample data:\n{schema_context}\n\nQuery: {user_query}\n\nPrevious Error:{previous_error}"
    else:
        user_content = f"Schema/sample data:\n{schema_context}\n\nQuery: {user_query}"
    
    raw_code = ask_llm(user_content, system_prompt)
    
    if raw_code.startswith("```python"):
        cleaned_code = raw_code.replace("```python", "").replace("```", "").strip()
    else:
        cleaned_code = raw_code
    
    return cleaned_code

### Validator LLM 

In [9]:
def validate_code(code: str, schema_context: str) -> str:
    system_prompt = """
    You are a code safety and correctness reviewer. You will be given pandas code and 
    sample data context. Check that:
    - The code only uses pandas operations on a DataFrame called `df`
    - It does not use file I/O, os, sys, subprocess, network calls, or exec/eval
    - It references column names that actually exist in the schema
    - It assigns a final answer to a variable called `result`
    
    Respond in EXACTLY this format, nothing else:
    "valid" if the code passes all checks
    "invalid: <short reason>" if it fails any check
    """
    
    user_content = f"Schema/sample data:\n{schema_context}\n\nCode to review:\n{code}"
    
    raw_response = ask_llm(user_content, system_prompt)
    
    cleaned = raw_response.strip().lower()
    
    return cleaned

In [10]:
def generate_validated_code(user_query: str, schema_context: str, max_retries: int = 3):
    previous_error = None
    
    for attempt in range(max_retries):
        code = generate_pandas_code(user_query, schema_context, previous_error)
        validation_result = validate_code(code, schema_context)
        
        if validation_result == "valid":
            return code
            
        print(f"Attempt {attempt + 1} failed: {validation_result}")
        previous_error = validation_result
    
    return previous_error

In [15]:
import pandas as pd
def execute_code(code: str, df):
    namespace = {"df": df, "pd": pd}
    exec(code, namespace)
    output = namespace["result"]
    return output

In [16]:
def format_answer(user_query: str, execution_result) -> str:
    system_prompt = "You are a helpful assistant. Given a user's question and the computed result, answer clearly and naturally in a sentence or two."
    user_content = f"Question: {user_query}\nComputed result:\n{execution_result}"
    return ask_llm(user_content, system_prompt)

In [26]:
def handle_query(user_query: str, has_document: bool = False, document_path:str = None):
    intent = classify_intent(user_query, has_document)
    
    if intent == "direct":
        answer = ask_llm(user_query)
        return answer
    elif intent == "document":
        document = load_document(document_path)
        schema = get_schema_context(document)
        output = generate_validated_code(user_query=user_query, schema_context=schema, max_retries=3)
        llm_input = execute_code(output, df=document)
        result = format_answer(user_query=user_query, execution_result=llm_input)
        return f"Question: {user_query} \n{result}"
    else:
        return "Sorry, I couldn't understand your request. Please rephrase."

# test
# print(handle_query("HI"))
# print(handle_query("Who wrote Romeo and Juliet?"))
# print(handle_query("What are the sales of the keyboard?", has_document=True, document_path="sales_data_large.xlsx"))
# print('\n\n')
print(handle_query("How many PCM have we sold in the past 1 year?", has_document=True, document_path="sales_data_large.xlsx"))
# print('\n\n')
# print(handle_query("How is Kiran performing?", has_document=True, document_path="sales_data_large.xlsx"))
# print('\n\n')
# print(handle_query("How are the sales of computer in East?", has_document=True, document_path="sales_data_large.xlsx"))
# print('\n\n')
# print(handle_query("Give me the sales by region? Also tell me the best sale for the past 2 years - which region it happened in and who is my best performer in that region", has_document=True, document_path="sales_data_large.xlsx"))

Question: How many PCM have we sold in the past 1 year? 
We haven't sold any PCM in the past year.
